# 第11回 演習：似た国を、グループに分ける


---


## 前回からの接続


前回は、392台の車を主成分（principal component）の地図に置いて、1台ずつ説明しました。でも、「似た車どうしを、グループに分けるには」という問いが残りました。

今日は、似たものどうしを、データからグループとして取り出す方法、クラスタリング（clustering）を学びます。正解のラベルはありません。何をもって「似ている」とするかを決めて、グループを見つけます。


---


## 今日の分析目標


あなたは国際協力機関の担当者です。142か国を、平均寿命と豊かさから、支援の方針が近いグループに分けたい。

k-meansという方法を自分で書いて中身を確かめ、3つのグループに分け、それぞれに名前と支援の方針をつけます。最後の「目標に答えられたか」で振り返りましょう。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- k-meansの手順（割り当て → 重心の更新 → 動かなくなるまで）を説明できる
- `while`で、条件が成り立つあいだ手順をくり返すコードを書ける
- 自分で書いたk-meansと、scikit-learnの`KMeans`の結果を比べられる
- グループを要約して名前をつけ、標準化（standardization）の有無で分け方が変わることを確かめられる
- グループの境目にいる国を見つけられる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ② の解析（グループに分ける）と、③ の解釈（名前と方針をつける）が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (2, 3)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. データを知る


グループに分ける前に、まず142か国がどう散らばっているかを見ます。Gapminderのデータです。世界の国々の、平均寿命（`lifeExp`）と一人あたりのGDP（`gdpPercap`、ドル）が、5年ごとに並んでいます。今日は、2007年の142か国を使います。


### お手本：2007年の国を取り出す

- 役割：データを読み込み、2007年だけを取り出し、GDPの対数の列を加える
- 入力：データの置き場所とファイル名
- 出力：表`d`（142か国）

GDPは、277ドルから49357ドルまで、180倍近い開きがあります。そのままでは、豊かな国の違いばかりが目立つので、`np.log10`で対数にします（第4回）。対数にすると、「10倍の違い」が同じ幅になります。


In [ ]:
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise_preview/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/gapminder.csv')
d = df[df['year'] == 2007].copy().reset_index(drop=True)
d['loggdp'] = np.log10(d['gdpPercap'])       # GDP の対数
print(len(d))
print(d[['lifeExp', 'loggdp']].describe().round(2))


142か国。平均寿命は39.6〜82.6歳、GDPの対数は2.44〜4.69です。


In [ ]:
# @title 図：142か国の平均寿命と豊かさ
plt.scatter(d['gdpPercap'], d['lifeExp'], s=25, alpha=0.7, color='#0066cc')
plt.xscale('log'); plt.xlabel('一人あたり GDP（ドル、対数目盛）')
plt.ylabel('平均寿命（歳）'); plt.show()


右上（豊かで長寿）に固まった国々と、左下（貧しく短命）に広がる国々が見えます。この散らばりを、いくつかのグループに分けたいのです。


### 変えてみる：標準化する

2つの列は、単位も散らばりの大きさも違います（平均寿命の標準偏差は12.07、GDPの対数は0.59）。距離で近さを測るので、標準化して物差しをそろえます（第3回）。


In [ ]:
X = StandardScaler().fit_transform(d[['lifeExp', 'loggdp']])
print(X.shape, X.mean(axis=0).round(2), X.std(axis=0).round(2))


142行×2列。どちらの列も、平均0・標準偏差（standard deviation）1になりました。


---


## 2. k-meansを、自分で書く


2つの列の物差しがそろいました。では、近い国どうしを、どうやってグループに集めればよいでしょうか。k-meansは、次の手順で点をK個のグループに分けます。

1. K個の中心を、どこかに置く
2. 各点を、いちばん近い中心のグループに割り当てる
3. 各グループの重心（点の平均の位置）に、中心を動かす
4. 中心が動かなくなるまで、2と3をくり返す

まず、2と3を1回ぶんだけ書いてみます。


### おさらい：for

```python
for i in range(3):
    print(i)          # 0, 1, 2 と表示される
```

`for`は、並びの中身を1つずつ名前に入れながら、字下げした行を繰り返す書き方でした。


### おさらい：[ ] の中のfor（内包表記）

```python
out = [i * 10 for i in range(3)]   # out は [0, 10, 20]
```

`[式 for 名前 in 並び]`と書くと、並びの中身を1つずつ名前に入れて式を計算し、その結果をリストに集めます（内包表記）。`out = []`から`for`の中で`append`していくのと同じことを、1行で書けます。


### お手本：割り当てと、中心の更新を1回

- 役割：最初の3か国の位置を中心にして、各国を近い中心に割り当て、中心を重心（centroid）に動かす
- 入力：標準化した`X`
- 出力：割り当て`labels`と、動かしたあとの中心`new`

`((X - c) ** 2).sum(axis=1)`は、中心`c`から142か国それぞれへの距離の2乗です。`[… for c in centers]`で3つの中心それぞれについて計算し、3本の並びをリストに集めます。それを`np.column_stack`で横に並べると、142行×3列の表になります。`argmin(axis=1)`は、行ごとにいちばん小さい列の番号、つまりいちばん近い中心の番号を返します。


In [ ]:
centers = X[[0, 1, 2]]                    # 最初の3か国の位置を中心に
dist = np.column_stack([((X - c) ** 2).sum(axis=1) for c in centers])
labels = dist.argmin(axis=1)              # いちばん近い中心の番号
new = np.array([X[labels == j].mean(axis=0) for j in range(3)])
print(np.bincount(labels))                # グループごとの国の数
print(new.round(2))


最初の3か国（アフガニスタン・アルバニア・アルジェリア）を中心にすると、1回目の割り当ては、グループごとの国の数がばらばらです。中心は、それぞれのグループの重心に動きました。

でも、中心が動けば、近い中心も変わります。だから、割り当てをやり直し、また中心を動かし……と、動かなくなるまでくり返す必要があります。何回くり返せばよいかは、前もってわかりません。


### 変えてみる：whileで、動かなくなるまでくり返す

`for`は、くり返す回数（並びの長さ）が決まっているときの書き方でした。回数がわからず、「ある条件が成り立つあいだ」くり返したいときは、`while`を使います。

```python
while 条件:
    くり返す手順
```

条件が`True`のあいだ、中の手順をくり返します。中で条件を変えないと、いつまでも止まらないので注意します。ここでは、「中心が動いた（`moved`が`True`）」あいだくり返し、動かなくなったら`moved`を`False`にして止めます。


### おさらい：np.allcloseとnot

```python
np.allclose([1.0, 2.0], [1.0, 2.0000001])   # True
not True                                     # False
```

`np.allclose(a, b)`は、2つの配列がどこも（計算の誤差を除いて）ほぼ等しければ`True`を返します。`not`は、`True`と`False`を入れかえます。だから`not np.allclose(new, centers)`は、「新しい中心が、前の中心と等しくない（動いた）」なら`True`になります。


In [ ]:
centers = X[[0, 1, 2]]
moved, n_iter = True, 0
while moved:                  # 中心が動いたあいだ、くり返す
    dist = np.column_stack([((X - c) ** 2).sum(axis=1)
                            for c in centers])
    labels = dist.argmin(axis=1)
    new = np.array([X[labels == j].mean(axis=0) for j in range(3)])
    moved = not np.allclose(new, centers)     # 動いたか
    centers = new
    n_iter += 1
print(n_iter, np.bincount(labels))


5回で中心が動かなくなりました。グループの国の数は46・47・49です。


---


## 3. KMeansで答え合わせ


k-meansを自分で書けました。では、その結果は合っているでしょうか。scikit-learnの`KMeans`と比べます。

### お手本：同じ出発点で、KMeansと比べる

- 役割：scikit-learnの`KMeans`を、自分のk-meansと同じ出発点（最初の3か国）で動かし、結果が一致するかを確かめる
- 入力：`X`、出発点
- 出力：2つの分け方の一致度

2つの分け方が同じかは、ARI（adjusted Rand index）で測ります。グループの番号の付け方が違っても、同じ国どうしが同じグループにいれば1、でたらめに分けたのと同じくらいなら0になります。


In [ ]:
km1 = KMeans(n_clusters=3, init=X[[0, 1, 2]], n_init=1).fit(X)
print(km1.n_iter_, adjusted_rand_score(labels, km1.labels_))


くり返しは同じ5回、ARIは1.0。完全に一致しました。`KMeans`の中身は、今書いた`while`のくり返しそのものです。


### 変えてみる：出発点を10通り試す

k-meansの答えは、出発点によって変わることがあります。`KMeans`に`n_init=10`を指定すると、乱数（random number）で選んだ出発点（初めの設定では、たがいに離れた点を選びやすい工夫つき）を10通り試し、いちばんよいものを残します。「よい」の物差しは慣性（inertia）です。各点から自分のグループの中心までの距離の2乗の合計で、小さいほどまとまっています。


In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
print(round(km1.inertia_, 2), round(km.inertia_, 2))
print(round(adjusted_rand_score(km1.labels_, km.labels_), 2))


慣性は、最初の3か国から出発すると54.67、10通り試すと53.75と小さくなりました。ARIは0.73で、分け方も変わりました。最初の3か国からの出発は、慣性のより大きい（より悪い）別の分け方で止まっていたのです。k-meansは、止まったところがいちばんよい分け方とは限りません。

これからは、10通り試した`km`の分け方を使います。


---


## 4. グループに名前をつける


分け方は、10通り試したものに決まりました。では、3つはそれぞれどんなグループでしょうか。クラスタリングの結果は、グループの番号（0・1・2）だけです。番号には意味がありません。中身を要約して、名前をつけます。


In [ ]:
# @title 図：3つのグループ
d['cluster'] = km.labels_
for cl, col in zip(range(3), ['#e63946', '#e9c46a', '#2a9d8f']):
    m = d['cluster'] == cl
    plt.scatter(d.loc[m, 'gdpPercap'], d.loc[m, 'lifeExp'], s=25,
                alpha=0.8, color=col, label=f'グループ{cl}')
plt.xscale('log'); plt.xlabel('一人あたり GDP（ドル、対数目盛）')
plt.ylabel('平均寿命（歳）'); plt.legend(); plt.show()


### TODO①：グループを要約する

`d['cluster'] = km.labels_`でグループの番号を表に加え、グループごとの国の数、平均寿命の平均、GDPの平均を表示してください。


In [ ]:
# TODO: グループごとの国の数・平均寿命・GDP の平均を表示する
# ヒント: d['cluster'] = km.labels_ のあと、
#         d.groupby('cluster').agg(国数=('country', 'size'), …)
...


グループ0は52か国・平均寿命53歳・GDP約2100ドル、グループ1は51か国・72歳・約7300ドル、グループ2は39か国・79歳・約3万ドル。順に「最貧」「新興」「先進」と名前がつけられます。


---


## 5. 標準化しないと？


3つのグループに、最貧・新興・先進と名前がつきました。ただし、この分け方は標準化した2列で求めたものです。1節で標準化したのは、2つの列の散らばりの大きさをそろえるためでした。標準化しないと、どうなるでしょうか。


### TODO②：標準化せずに分けて、比べる

`d[['lifeExp', 'loggdp']].values`（標準化しない値）を`KMeans(n_clusters=3, n_init=10, random_state=42)`で分け、`km.labels_`とのARIと、グループごとの平均寿命・GDPの平均を表示してください。


In [ ]:
# TODO: 標準化しない値で分けて、ARI とグループの要約を表示する
# ヒント: raw = KMeans(…).fit_predict(d[['lifeExp', 'loggdp']].values)
#         adjusted_rand_score(km.labels_, raw)
#         d.groupby(raw)[…].mean()
...


ARIは0.41。分け方が大きく変わりました。グループの平均を見ると、平均寿命は48・61・76歳ときれいに分かれるのに、GDPは約2200ドルと約2500ドルの2つのグループがほとんど同じです。平均寿命だけで分けたようなグループです。

平均寿命の散らばり（標準偏差12）は、GDPの対数（0.59）の20倍もあるので、距離はほとんど平均寿命の差で決まってしまいます。距離で分ける方法では、標準化で物差しをそろえることが欠かせません。


---


## 6. 支援の方針のためのグループ


標準化した分け方を使う理由が分かりました。では、最初の問いに戻ります。3つのグループを、支援の方針の言葉で読み直します。まず、各グループがどの大陸の国でできているかを見ます。


### お手本：グループと大陸のクロス集計

- 役割：グループと大陸の組み合わせごとに、国の数を数える
- 入力：`d['cluster']`、`d['continent']`
- 出力：3行×5列の表


In [ ]:
d['cluster'] = km.labels_
print(pd.crosstab(d['cluster'], d['continent']))


最貧グループ（0）の52か国のうち44か国がアフリカ、先進グループ（2）の39か国のうち23か国がヨーロッパです。新興グループ（1）は、アメリカ大陸とアジアを中心に、いろいろな大陸の国が混ざります。グループは大陸とかなり重なりますが、同じではありません。


### TODO③：気になる国は、どのグループか

日本・中国・インド・南アフリカ・ボツワナが、どのグループにいるかを、平均寿命とGDPとあわせて表示してください。


In [ ]:
# TODO: 5か国のグループと、平均寿命・GDP を表示する
# ヒント: pick = ['Japan', 'China', 'India',
#                 'South Africa', 'Botswana']
#         d[d['country'].isin(pick)]
...


日本は先進（2）、中国とインドは新興（1）です。南アフリカとボツワナは、GDPは約9300ドル・約1万2600ドルと新興グループ並みなのに、平均寿命が49歳・51歳と短いため、最貧グループ（0）に入っています。HIVの流行の影響です。GDPだけで見ていたら、見落とすところでした。


### TODO④：境目にいる国を探す

`km.transform(X)`は、各国から3つの中心までの距離を返します。行ごとに小さい順に並べ（`np.sort(…, axis=1)`）、「2番目に近い中心までの距離 − いちばん近い中心までの距離」が小さい5か国を表示してください。この差が小さい国は、2つのグループの境目にいます。


In [ ]:
# TODO: 2つの中心までの距離の差が小さい5か国を表示する
# ヒント: dist = np.sort(km.transform(X), axis=1)
#         d['margin'] = dist[:, 1] - dist[:, 0]
#         → sort_values('margin')
...


いちばん境目にいるのは、トリニダード・トバゴ（差0.03）、イラク、北朝鮮、アルゼンチン、メキシコです。中心を少し動かすだけで、別のグループに移りうる国です。こうした国は、グループの方針をそのまま当てはめず、1か国ずつ確かめる必要があります。

担当者としての答えは、次のとおりです。

> 142か国は、平均寿命と豊かさから、最貧（52か国）・新興（51か国）・先進（39か国）の3グループに分けられる。最貧グループは保健と経済の両面で重点的に支援し、新興グループは経済の底上げを中心に支援する。南アフリカやボツワナのように、豊かさの割に寿命の短い国は最貧グループに入ることに注意し、境目の国は1か国ずつ確かめる。

まだ言えないことは、次の2つです。

- 3つでよいのか：K＝3は、こちらで決めた数。4つや5つのほうが、よい分け方ではないのか
- 分け方の手法：k-meansのほかの方法で分けたら、同じグループになるか（課題）


---


## 目標に答えられたか


- 今日の目標は「142か国を、支援の方針が近いグループに分ける」ことでした。6節の答えで、3つのグループにどんな名前と方針をつけましたか？
- 2節の`while`は、どんな条件のあいだくり返し、何回で止まりましたか？
- 3節で、最初の3か国から出発した分け方と、10通り試した分け方は、慣性とARIがどう違いましたか？
- TODO②で、標準化しないと、どんなグループになりましたか？ それはなぜですか？


---


## 今日の要点


- k-meansは、「近い中心に割り当てる → 中心を重心に動かす」を、動かなくなるまでくり返す
- 答えは出発点で変わる。`n_init`で何通りも試し、慣性のいちばん小さいものを残す
- グループの番号には意味がない。要約して名前をつけ、方針に結びつける
- 距離で分けるので、標準化で物差しをそろえる。そろえないと、散らばりの大きい列だけで分けてしまう
- 2つの分け方の一致度はARIで測る（1で一致、0でたらめ並み）
- Python：`while 条件:`で、条件が成り立つあいだくり返す。回数がわからないくり返しに使う


---


## 次回へ


次回は、実際の研究で使われているコードを読みます。今日書いたような手順が、何本ものファイルや関数に分かれて組み込まれている様子を見ます。論文のコードを読んで動かし、論文の主張を手元で確かめられるでしょうか。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

k-meansのほかに、階層クラスタリングという方法があります。はじめは1か国ずつを1つのグループとし、グループどうしを1組ずつ順につないでいき、指定した数のグループになったら止めます。どの組をつなぐかの基準は、初めの設定ではウォード法です。つないでもグループの中のまとまり（慣性）がいちばん崩れない組から、つなぎます。

`from sklearn.cluster import AgglomerativeClustering`を読み込み、`AgglomerativeClustering(n_clusters=3)`で同じ`X`を3つに分けて、`d['hc']`に入れてください。そして、グループごとの国の数・平均寿命・GDPの平均を表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`hc = AgglomerativeClustering(n_clusters=3)`を作り、`d['hc'] = hc.fit_predict(X)`とします。
要約はTODO①と同じく`d.groupby('hc').agg(…)`です。

</details>


### 応用②（判断）

応用①の階層クラスタリングの分け方と、k-meansの分け方（`km.labels_`）のARIはいくつですか。小数第2位まで答えてください（例: 0.57）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

3節と同じ`adjusted_rand_score`に、`km.labels_`と`d['hc']`を渡します。`pd.crosstab(km.labels_, d['hc'])`で、どのグループの何か国が違ったかも見られます。

</details>


### 応用③（解釈）

k-meansと階層クラスタリングは、多くの国で同じ分け方をしましたが、一部の国で割り当てが違いました。応用②のARIと、2つの要約（どのグループが増え、どのグループが減ったか）をふまえて、報告書にどちらの結果を載せるかとその理由を、国際協力の担当者に向けて3行程度で書いてください。


（ここに3行程度で書く）


---


## 発展（任意）


### HDBSCAN：Kを決めず、まとまらない国は外れ値にする

k-meansでは、人が決めることが2つありました。Kを先に決めることと、すべての国をどこかのグループに必ず入れることです。境目の国も、どこかのグループに無理に入れられます。

HDBSCANは、点が密に集まっている場所をグループとし、まばらな場所の点は外れ値（ラベル −1）として残す方法です。Kは決めません。決めるのは「グループと認める最小の国の数」（`min_cluster_size`）だけです。


In [ ]:
from sklearn.cluster import HDBSCAN

hdb = HDBSCAN(min_cluster_size=8, copy=True).fit_predict(X)
d['hdb'] = hdb
print('グループの数:', hdb.max() + 1, ' 外れ値:', (hdb == -1).sum(), 'か国')
print(d.groupby('hdb').agg(国数=('country', 'size'),
                           平均寿命=('lifeExp', 'mean'),
                           GDP=('gdpPercap', 'mean')).round(0))


In [ ]:
for g in sorted(d['hdb'].unique()):
    m = d['hdb'] == g
    label = '外れ値' if g == -1 else f'グループ{g}'
    plt.scatter(d.loc[m, 'gdpPercap'], d.loc[m, 'lifeExp'], s=25,
                marker='x' if g == -1 else 'o', label=label)
plt.xscale('log'); plt.xlabel('一人あたり GDP（ドル、対数目盛）')
plt.ylabel('平均寿命（歳）'); plt.legend(); plt.show()


読み方：グループは3つ（30・39・32か国）と、41か国の外れ値になりました。3つのグループは、k-meansの最貧・新興・先進の「芯」にあたります。

×印の外れ値には、2種類の国が混ざっています。1つは、図の左下のまばらな場所にいる国です。アンゴラ、ブルンジ、ジンバブエ、コンゴ民主共和国など、平均寿命の短い最貧の国々は、たがいに離れて散らばっているので、密なグループになりません。GDPの割に平均寿命の短いボツワナや南アフリカも、まばらな場所にいます。もう1つは、グループのあいだの境目にいる国で、チリ、コスタリカ、イラク、トリニダード・トバゴなどです。

「どこにも無理に入れない」ことで、境目の国だけでなく、まばらな場所にいる国も、1か国ずつ確かめるべき国として見えてきます。

試すなら、`min_cluster_size`を5や15に変えて、グループの数と外れ値の数がどう変わるか見てみましょう。
